# Dummy Agent Library

Creating an agent from scratch.

In [ ]:
%pip install -q huggingface_hub -U
%pip install -q dotenv -U

## Serverless API
In the Hugging Face ecosystem, there is a convenient feature called Serverless API that allows you to easily run inference on many models. There's no installation or deployment required.

In [11]:
import os
# from huggingface_hub import InferenceClient
import dotenv
from openai import OpenAI

dotenv.load_dotenv()

HF_TOKEN = os.environ.get("HF_TOKEN")

# Not using the Hugging Face InferenceClient because HF token requires a paid plan.
# Instead, we will use the local LLM model running with Ollama server.
# client = InferenceClient(model="moonshotai/Kimi-K2.5", token=HF_TOKEN)
client = OpenAI(
    base_url="http://localhost:11434/v1", 
    api_key="ollama", # Ollama does not require a real API key
)

Using the `chat` method since it is a convenient and reliable way to apply chat templates.

In [15]:
output = client.chat.completions.create(
    model="qwen2:7b",
    messages=[
        {"role": "user", "content": "The capital of Finland is"},
    ],
    stream=False,
    max_tokens=1024,
    extra_body={'thinking': {'type': 'disabled'}},
)
print(output.choices[0].message.content)

Helsinki.


In [16]:
# This system prompt is a bit more complex and actually contains the function description already appended.
# Here we suppose that the textual description of the tools has already been appended.

SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {{"location": "New York"}}
}}


ALWAYS use the following format:

Question: the input question you must answer

Thought: you should always think about one action to take. Only one action at a time in this format:

Action:
$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.

... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is the current weather in London?"},
]

print(messages)

[{'role': 'system', 'content': 'Answer the following questions as best you can. You have access to the following tools:\n\nget_weather: Get the current weather in a given location\n\nThe way you use the tools is by specifying a json blob.\nSpecifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).\n\nThe only values that should be in the "action" field are:\nget_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}\nexample use :\n\n{{\n  "action": "get_weather",\n  "action_input": {{"location": "New York"}}\n}}\n\n\nALWAYS use the following format:\n\nQuestion: the input question you must answer\n\nThought: you should always think about one action to take. Only one action at a time in this format:\n\nAction:\n$JSON_BLOB (inside markdown cell)\n\nObservation: the result of the action. This Observation is unique, complete, and the source of truth.\n\n... 

In [19]:
output = client.chat.completions.create(
    model="qwen2:7b",
    messages=messages,
    stream=False,
    max_tokens=200,
    extra_body={'thinking': {'type': 'disabled'}},
)
print(output.choices[0].message.content)

Thought: To find out the current weather in London, I need to utilize the get_weather tool, specifying London as the location.

Action:
```
{{
  "action": "get_weather",
  "action_input": {{"location": {"type": "string", "value": "London"}}}
}}
```


In [21]:
# The answer was hallucinated by the model. We need to stop to actually execute the function!
output = client.chat.completions.create(
    model="qwen2:7b",
    messages=messages,
    max_tokens=150,
    stop=["Observation:"], # Let's stop before any actual function is called
    extra_body={'thinking': {'type': 'disabled'}},
)

print(output.choices[0].message.content)

Thought: I need to use the get_weather action to get the current weather in London.

Action:
```
{
  "action": "get_weather",
  "action_input": {"location": "London"}
}
```




Now, creating the dummy get weather function:

In [22]:
def get_weather(location: str) -> str:
    return f"The current weather in {location} is sunny with a temperature of 25°C.\n"

get_weather("London")

'The current weather in London is sunny with a temperature of 25°C.\n'

Concatenating the system prompt with the user prompt to create a single message for the model to process:

In [24]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
    {"role": "assistant", "content": output.choices[0].message.content + "Observation:\n" + get_weather('London')},
]

output = client.chat.completions.create(
    model="qwen2:7b",
    messages=messages,
    stream=False,
    max_tokens=200,
    extra_body={'thinking': {'type': 'disabled'}},
)

print(output.choices[0].message.content)

Thought: I now know the final answer
Final Answer: The current weather in London is sunny with a temperature of 25°C.
